# 🎚️ Notebook 2: Gradual Traffic Shift

Route, say, 10% of `/users` requests to the new service, 90% to legacy. Compare results, then ramp up. This is sometimes called **canary routing**.

## 🛠️ Setup

```bash
cd 05-microservices/strangler
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


In [ ]:
import random, collections

def legacy(path): return {'source':'legacy', 'path':path, 'v':1}
def new(path):    return {'source':'new',    'path':path, 'v':2}

class Router:
    def __init__(self):
        self.percent_new = {}  # path prefix -> int 0..100
    def set(self, prefix, pct):
        self.percent_new[prefix] = pct
    def handle(self, path):
        for prefix, pct in self.percent_new.items():
            if path.startswith(prefix):
                return new(path) if random.randint(1,100) <= pct else legacy(path)
        return legacy(path)

random.seed(0)
r = Router()
r.set('/users', 10)  # 10% canary

counts = collections.Counter(r.handle('/users/42')['source'] for _ in range(1000))
print('10% new:', counts)

r.set('/users', 50)  # ramp up
counts = collections.Counter(r.handle('/users/42')['source'] for _ in range(1000))
print('50% new:', counts)

r.set('/users', 100) # done
counts = collections.Counter(r.handle('/users/42')['source'] for _ in range(1000))
print('100% new:', counts)


### Production checklist
- ✅ **Dark launch** the new service: send traffic to *both*, compare responses.
- ✅ Watch **error rate + latency** before ramping.
- ✅ Keep a **rollback switch** to 0% if the canary smells bad.
- ✅ Migrate **data** carefully — often dual-write during the transition.